In [2]:
from flask import Flask, request, jsonify
import whisper
import os
import traceback
from werkzeug.utils import secure_filename
import uuid
import time
from datetime import timedelta
import sys
import torch
import warnings
warnings.filterwarnings("ignore")

app = Flask(__name__)

# 配置上传文件夹
UPLOAD_FOLDER = 'uploads'
if not os.path.exists(UPLOAD_FOLDER):
    os.makedirs(UPLOAD_FOLDER)

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 限制上传文件大小为16MB

# 设置ffmpeg路径
os.environ['PATH'] = os.environ['PATH'] + ';C:\\Tools\\ffmpeg\\bin'  # 请根据您的实际安装路径修改

# 强制使用CPU并清理GPU内存
torch.cuda.empty_cache()
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # 禁用GPU
device = "cpu"
print("使用CPU模式运行")

print("正在加载Whisper模型...")
# 加载Whisper模型（使用最大的large-v3模型）
model = whisper.load_model("medium", device="cpu")
print("Whisper模型加载完成！")

def format_timestamp(seconds):
    """将秒数转换为 [mm:ss.xx] 格式"""
    return f"[{str(timedelta(seconds=seconds)).split('.')[0]}.{str(seconds).split('.')[-1][:2]}]"

def generate_lyrics_with_timestamps(result):
    """生成带时间戳的歌词"""
    lyrics = []
    
    # 处理每个片段
    for segment in result['segments']:
        start_time = format_timestamp(segment['start'])
        text = segment['text'].strip()
        if text:  # 只添加非空行
            lyrics.append(f"{start_time}{text}")
    
    return '\n'.join(lyrics)

@app.route('/transcribe', methods=['POST'])
def transcribe_audio():
    if 'file' not in request.files:
        return jsonify({'error': '没有文件上传'}), 400
    
    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': '没有选择文件'}), 400
    
    if file:
        # 生成唯一的文件名
        original_filename = secure_filename(file.filename)
        if not original_filename:
            original_filename = 'audio'
        file_extension = os.path.splitext(original_filename)[1]
        if not file_extension:
            file_extension = '.mp3'
        
        unique_filename = f"{uuid.uuid4()}{file_extension}"
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], unique_filename)
        
        try:
            print(f"接收到文件: {original_filename}")
            # 保存文件
            file.save(filepath)
            print("文件保存成功，开始转写...")
            
            start_time = time.time()
            # 使用Whisper进行转写，指定中文，并添加音频处理参数
            result = model.transcribe(
                filepath,
                language='zh',
                # 音频处理参数
                initial_prompt="这是一首中文歌曲，请识别歌词。歌词通常有韵律和重复性，每句歌词都有特定的节奏和押韵。",  # 更详细的提示
                condition_on_previous_text=False,  # 关闭上下文依赖，避免错误累积
                temperature=0.0,  # 降低随机性
                compression_ratio_threshold=2.4,  # 控制输出长度
                logprob_threshold=-1.0,  # 降低置信度阈值
                no_speech_threshold=0.6,  # 提高无语音检测阈值
                word_timestamps=True,  # 启用词级别时间戳
                fp16=False  # 强制使用FP32
            )
            end_time = time.time()
            
            print(f"转写完成！耗时: {end_time - start_time:.2f} 秒")
            
            # 生成带时间戳的歌词
            lyrics = generate_lyrics_with_timestamps(result)
            
            # 删除临时文件
            if os.path.exists(filepath):
                os.remove(filepath)
                print("临时文件已清理")
            
            return jsonify({
                'success': True,
                'lyrics': lyrics,
                'time_taken': f"{end_time - start_time:.2f}秒",
                'device': device
            })
        except Exception as e:
            # 获取详细的错误信息
            error_details = traceback.format_exc()
            print(f"错误详情: {error_details}")
            
            # 确保在发生错误时也删除临时文件
            if os.path.exists(filepath):
                os.remove(filepath)
                print("发生错误，临时文件已清理")
                
            return jsonify({
                'error': str(e),
                'details': error_details
            }), 500

if __name__ == '__main__':
    print("服务启动中...")
    print("请等待Whisper模型加载完成")
    
    # 检查端口是否被占用
    port = 5000
    try:
        app.run(host='0.0.0.0', port=port)
    except Exception as e:
        print(f"启动服务时出错: {str(e)}")
        sys.exit(1)

使用CPU模式运行
正在加载Whisper模型...
Whisper模型加载完成！
服务启动中...
请等待Whisper模型加载完成
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.108.2:5000
Press CTRL+C to quit


接收到文件: -.mp3
文件保存成功，开始转写...


127.0.0.1 - - [28/Apr/2025 23:46:14] "POST /transcribe HTTP/1.1" 200 -


转写完成！耗时: 282.30 秒
临时文件已清理
